<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-5_Rule_Based_Policy_Harness_Test_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 5. Rule-Based Policy로 Agent Harness 테스트

**Google Colab / Python 3 / API 키 및 패키지 설치 불필요**

### 학습 목표
1. `run_agent()`가 상태(State), 도구(Tools), 반복 제한을 관리하는 과정을 이해한다.
2. `mock_policy()`가 **LLM 대신 미리 정해진 규칙**으로 다음 도구를 선택하게 한다.
3. 정상 종료(`completed`), 허용되지 않은 도구(`invalid_tool`), 최대 단계(`max_steps`)를 직접 관찰한다.

> **중요:** 이 실습은 Agent **Harness를 테스트**하는 것입니다. `mock_policy()`가 규칙 기반이므로 그 자체를 LLM Agent라고 부르지 않습니다.

## 1. 실행 흐름

`목표/상태 → mock_policy(행동 선택) → Harness(검증·도구 실행) → 관찰 결과 저장 → 정책 재호출`

- `tool == "finish"`이면 종료합니다.
- 알 수 없는 도구를 요청하면 안전하게 중단합니다.
- `max_steps`를 초과하지 않고 종료합니다.

**참고:** 이미지의 예제를 바탕으로 코드를 구성하고, 학생이 실행 중단 원인을 쉽게 알아보도록 일부 입력 검증을 추가했습니다.

## 2. 검색 도구 만들기

In [1]:
def search_course(query):
    """수업 정보를 문자열로 검색하는 모의 도구입니다."""
    # 간단한 검색 도구: 포함 여부만 검사합니다.
    if "평가" in query:
        return "과제20 중간30 기말30 출석20"
    if "AI" in query:
        return "AI 활용 부분 허용, 출처 명시"
    return "no result"

# 사용 가능한 도구를 이름과 함수로 연결합니다.
tools = {"search": search_course}

print(search_course("지능시스템 평가"))
print(search_course("지능시스템 AI 정책"))

과제20 중간30 기말30 출석20
AI 활용 부분 허용, 출처 명시


## 3. Harness 함수 구현

In [3]:
def run_agent(goal, policy, tools, max_steps=8):
    """Policy의 선택에 따라 도구를 실행하는 간단한 Harness입니다."""
    # 현재 목표와 지금까지 관찰한 도구 결과를 저장합니다.
    state = {"goal": goal, "observations": []}
    log = []  # 실행 과정을 기록합니다.

    for step in range(1, max_steps + 1):
        # 1) 현재 상태를 보고 다음 행동을 선택합니다.
        action = policy(state)

        # 2) 행동 형식이 잘못되었는지 확인합니다.
        if not isinstance(action, dict) or not isinstance(action.get("tool"), str):
            log.append({"step": step, "error": "invalid_action", "action": action})
            return None, log, "invalid_action"

        tool_name = action["tool"]
        args = action.get("args", {})

        # 3) 완료 요청이면 최종 답변을 반환합니다.
        if tool_name == "finish":
            answer = action.get("answer")
            if not isinstance(answer, str) or not answer.strip():
                log.append({"step": step, "error": "invalid_answer"})
                return None, log, "invalid_answer"
            log.append({"step": step, "tool": "finish", "answer": answer})
            return answer, log, "completed"

        # 4) 등록되지 않은 도구는 실행하지 않습니다.
        if tool_name not in tools:
            log.append({"step": step, "tool": tool_name, "error": "invalid_tool"})
            return None, log, "invalid_tool"

        # 5) 예제에서는 모든 도구 인자가 딕셔너리여야 합니다.
        if not isinstance(args, dict):
            log.append({"step": step, "tool": tool_name, "error": "invalid_args"})
            return None, log, "invalid_args"

        # 6) 도구를 실행하고 반환값을 관찰 결과로 저장합니다.
        try:
            result = tools[tool_name](**args)
        except (TypeError, ValueError) as error:
            log.append({"step": step, "tool": tool_name, "error": str(error)})
            return None, log, "tool_error"

        log.append({"step": step, "tool": tool_name, "args": args, "result": result})
        state["observations"].append({"action": action, "result": result})

    # 지정된 횟수 내에 finish가 나오지 않으면 종료합니다.
    return None, log, "max_steps"

## 4. 규칙 기반 Policy 만들기

- 관찰이 없으면 `평가` 정보를 검색합니다.
- `AI 활용` 문구를 아직 관찰하지 못했다면 `AI 정책`을 검색합니다.
- 두 정보가 확보되면 `finish`를 반환합니다.

**주의:** `AI 활용` 문구 검사는 이미지 속 예제를 따릅니다. 실제로 더 신뢰성 있는 조건이 필요하면 정책을 수정할 수 있습니다.

In [4]:
def mock_policy(state):
    """LLM 없이 조건문으로 다음 행동을 정합니다."""
    observations = state["observations"]

    # 첫 단계: 무조건 평가 정보를 검색합니다.
    if not observations:
        return {
            "tool": "search",
            "args": {"query": "지능시스템 평가"},
        }

    # 지금까지 검색된 도구 결과들을 살펴봅니다.
    texts = [str(item["result"]) for item in observations]

    # AI 활용 정책을 아직 발견하지 못했다면 추가 검색합니다.
    if not any("AI 활용" in text for text in texts):
        return {
            "tool": "search",
            "args": {"query": "지능시스템 AI 정책"},
        }

    # 필요한 정보를 얻었으므로 완료를 선언합니다.
    return {
        "tool": "finish",
        "answer": "평가와 AI 정책 근거를 모두 확보했습니다.",
    }

## 5. 정상 실행: 도구 2회 사용 후 종료

In [5]:
answer, log, stop_reason = run_agent(
    goal="평가와 AI 정책을 조사하라.",
    policy=mock_policy,
    tools=tools,
)

print("종료 상태:", stop_reason)
print("최종 답변:", answer)
print("\n--- 실행 기록 ---")
for row in log:
    print(row)

assert stop_reason == "completed"
assert [row["tool"] for row in log] == ["search", "search", "finish"]
print("\n✅ 정상 완료 및 로그 검증 통과")

종료 상태: completed
최종 답변: 평가와 AI 정책 근거를 모두 확보했습니다.

--- 실행 기록 ---
{'step': 1, 'tool': 'search', 'args': {'query': '지능시스템 평가'}, 'result': '과제20 중간30 기말30 출석20'}
{'step': 2, 'tool': 'search', 'args': {'query': '지능시스템 AI 정책'}, 'result': 'AI 활용 부분 허용, 출처 명시'}
{'step': 3, 'tool': 'finish', 'answer': '평가와 AI 정책 근거를 모두 확보했습니다.'}

✅ 정상 완료 및 로그 검증 통과


### 관찰 포인트

**Step 1**의 `search` 결과는 평가 비율, **Step 2**는 AI 활용 정책입니다. **Step 3**에서 도구 호출 대신 `finish`를 반환합니다.

정책은 고정 규칙이지만, Harness는 이전 단계의 도구 결과를 `state["observations"]`로 전달합니다.

## 6. 경계 조건 테스트 ① 최대 단계 제한

In [6]:
# 항상 같은 도구를 선택하고 finish하지 않는 정책입니다.
def endless_policy(state):
    return {"tool": "search", "args": {"query": "지능시스템 평가"}}

answer2, log2, status2 = run_agent(
    goal="종료하지 않는 정책 테스트",
    policy=endless_policy,
    tools=tools,
    max_steps=3,
)
print("종료 상태:", status2)
print("실행 단계 수:", len(log2))
assert status2 == "max_steps" and len(log2) == 3
print("✅ 최대 3회만 실행됨")

종료 상태: max_steps
실행 단계 수: 3
✅ 최대 3회만 실행됨


## 7. 경계 조건 테스트 ② 등록되지 않은 도구 차단

In [7]:
def bad_tool_policy(state):
    # tools 목록에 없는 도구 이름을 요청합니다.
    return {"tool": "delete_all_files", "args": {}}

answer3, log3, status3 = run_agent(
    goal="허용되지 않은 도구 테스트",
    policy=bad_tool_policy,
    tools=tools,
)
print("종료 상태:", status3)
print("실행 기록:", log3)
assert status3 == "invalid_tool"
print("✅ 허용되지 않은 도구 실행 차단")

종료 상태: invalid_tool
실행 기록: [{'step': 1, 'tool': 'delete_all_files', 'error': 'invalid_tool'}]
✅ 허용되지 않은 도구 실행 차단


## 8. 경계 조건 테스트 ③ 잘못된 응답 형식 차단

In [8]:
def bad_format_policy(state):
    return {"action": "search", "args": {"query": "평가"}}  # tool 키가 없음

answer4, log4, status4 = run_agent(
    goal="잘못된 행동 형식 테스트",
    policy=bad_format_policy,
    tools=tools,
)
print("종료 상태:", status4)
print("실행 기록:", log4)
assert status4 == "invalid_action"
print("✅ JSON/딕셔너리 인터페이스 규칙 검증")

종료 상태: invalid_action
실행 기록: [{'step': 1, 'error': 'invalid_action', 'action': {'action': 'search', 'args': {'query': '평가'}}}]
✅ JSON/딕셔너리 인터페이스 규칙 검증


## 9. 학생 실습 과제

1. `search_course()`에 `"시험"` 검색 조건을 추가해 보세요.
2. `mock_policy()`를 수정하여 `평가 → AI → 시험` 순서로 검색한 뒤 완료하게 만드세요.
3. `max_steps=2`로 변경한 뒤 결과가 `completed`인지 `max_steps`인지 설명하세요.
4. `"args": {"wrong": "평가"}`를 반환하도록 수정하고 `tool_error`가 발생하는 이유를 확인하세요.

### 정리

- **Policy:** 다음 행동을 결정하는 로직(이번 실습에서는 규칙 기반)
- **Tools:** 실행할 수 있는 기능
- **Harness:** 상태, 도구 실행, 로그, 종료 조건, 예외 처리를 관리

**핵심:** 규칙 기반 정책으로 Harness를 시험할 수 있지만, 이것만으로 LLM이 행동을 자율적으로 결정하는 Agent가 되는 것은 아닙니다.